# Modelo digitalidad del cliente

In [0]:
!pip install kmodes scikit-learn-extra imageio

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import random

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import col, lit, udf
from pyspark.sql.window import Window

In [0]:
spark = SparkSession.builder.appName("Modelo Digitalidad del Cliente").getOrCreate()

In [0]:
df = spark.read.table("workspace.default.transaccional_clientes")
display(df.limit(10))

In [0]:
id_cols = ['cliente_id', 'agencia_id', 'ruta_id']

cat_cols = ['pais', 'region_comercial', 'tipo_cliente', 'madurez_digital', 'frecuencia_visitas', 'canal_pedido']

disc_cols = ['estrellas', 'materiales_distintos']

num_cols = ['facturacion_usd', 'cajas_fisicas']

date_cols = ['fecha_pedido_dt']

## Análisis exploratorio de datos

In [0]:
cliente_canal = (
    df
    .groupBy("cliente_id", "canal_pedido")
    .agg(f.count("*").alias("count"))
)

total_por_cliente = df.groupBy("cliente_id").agg(f.count("*").alias("total"))

cliente_canal = (
    cliente_canal
    .join(total_por_cliente, on="cliente_id", how="left")
    .withColumn("percentage", col("count") / col("total"))
    .select("cliente_id", "canal_pedido", "count", "percentage")
)

# Convert to pandas for plotting
cliente_canal_pd = cliente_canal.toPandas()

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=cliente_canal_pd,
    x="percentage",
    hue="canal_pedido"
)
plt.title("Histograma del porcentaje por canal_pedido")
plt.xlabel("Porcentaje")
plt.ylabel("Frecuencia")
plt.legend(title="Canal Pedido")
plt.show()

In [0]:
df_ym = df.withColumn(
    "year_month",
    f.date_format(col("fecha_pedido"), "yyyyMM").cast("int")
)

cliente_min_max = (
    df_ym.groupBy("cliente_id")
    .agg(
        f.min("year_month").alias("min_year_month"),
        f.max("year_month").alias("max_year_month"),
        f.countDistinct("year_month").alias("meses_activo")
    )
)

cliente_min_max = (
    cliente_min_max
    .withColumn("min_date", f.to_date(f.concat_ws("-", col("min_year_month").cast("string").substr(1,4), col("min_year_month").cast("string").substr(5,2), f.lit("01"))))
    .withColumn("max_date", f.to_date(f.concat_ws("-", col("max_year_month").cast("string").substr(1,4), col("max_year_month").cast("string").substr(5,2), f.lit("01"))))
    .withColumn("meses_total", f.months_between(col("max_date"), col("min_date")) + 1)
    .withColumn("meses_total", col("meses_total").cast("int"))
    .select("cliente_id", "min_year_month", "max_year_month", "meses_total", "meses_activo")
)

display(cliente_min_max.limit(20))

In [0]:
clientes_all = [row['cliente_id'] for row in df.select("cliente_id").distinct().collect()]
n_clients = min(20, len(clientes_all))
clientes_sample = random.sample(clientes_all, n_clients)

wind_cliente_seq = Window.partitionBy("cliente_id").orderBy("fecha_pedido")

df_seq = (
    df.filter(col("cliente_id").isin(clientes_sample))
    .withColumn("nro_pedido", F.row_number().over(wind_cliente_seq))
    .withColumn("es_digital", (col("canal_pedido") == "DIGITAL").cast("int"))
    .select("cliente_id", "nro_pedido", "es_digital")
    .orderBy("cliente_id", "nro_pedido")
)

df_seq_pd = df_seq.toPandas()

num_clientes = len(clientes_sample)
fig, axes = plt.subplots(num_clientes, 1, figsize=(12, 3*num_clientes), sharex=True)

if num_clientes == 1:
    axes = [axes]

for ax, cliente in zip(axes, clientes_sample):
    data = df_seq_pd[df_seq_pd['cliente_id'] == cliente]
    ax.plot(
        data['nro_pedido'],
        data['es_digital'],
        marker='o',
        drawstyle='steps-mid',
        color='tab:blue'
    )
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['NO DIGITAL', 'DIGITAL'])
    ax.set_title(f'Cliente {cliente}')
    ax.set_ylabel('Canal Digital')

axes[-1].set_xlabel('Número de Pedido')
plt.tight_layout()
plt.show()

In [0]:
df_box = (
    df.withColumn("es_digital", (col("canal_pedido") == "DIGITAL").cast("string"))
    .select("facturacion_usd", "cajas_fisicas", "materiales_distintos", "es_digital")
)

df_box_pd = df_box.toPandas()

fig, axes = plt.subplots(3, 1, figsize=(10, 18))

sns.boxplot(
    data=df_box_pd,
    x="es_digital",
    y="facturacion_usd",
    ax=axes[0]
    
)
axes[0].set_title("Facturación USD vs Digital/No Digital")
axes[0].set_xlabel("¿Digital?")
axes[0].set_ylabel("Facturación USD")

sns.boxplot(
    data=df_box_pd,
    x="es_digital",
    y="cajas_fisicas",
    ax=axes[1]
)
axes[1].set_title("Cajas Físicas vs Digital/No Digital")
axes[1].set_xlabel("¿Digital?")
axes[1].set_ylabel("Cajas Físicas")

sns.boxplot(
    data=df_box_pd,
    x="es_digital",
    y="materiales_distintos",
    ax=axes[2]
)
axes[2].set_title("Materiales Distintos vs Digital/No Digital")
axes[2].set_xlabel("¿Digital?")
axes[2].set_ylabel("Materiales Distintos")

plt.tight_layout()
plt.show()

In [0]:
clientes_all = [row['cliente_id'] for row in df.select("cliente_id").distinct().collect()]
n_clients = min(20, len(clientes_all))
clientes_sample = random.sample(clientes_all, n_clients)

wind_cliente_all = Window.partitionBy("cliente_id").orderBy("fecha_pedido")
wind_cliente_6p = wind_cliente_all.rowsBetween(-5, Window.currentRow)

df_seq = (
    df.filter(col("cliente_id").isin(clientes_sample))
    .withColumn("nro_pedido", F.row_number().over(wind_cliente_seq))
    .withColumn("es_digital", (col("canal_pedido") == "DIGITAL").cast("int"))
    .withColumn("acum_digital", F.sum("es_digital").over(wind_cliente_6p))
    .withColumn("proporcion_digital_hist", col("acum_digital") / col("nro_pedido"))
    .select("cliente_id", "nro_pedido", "proporcion_digital_hist", "es_digital")
    .orderBy("cliente_id", "nro_pedido")
)

df_seq_pd = df_seq.toPandas()

num_clientes = len(clientes_sample)
fig, axes = plt.subplots(num_clientes, 1, figsize=(12, 3*num_clientes), sharex=True)

if num_clientes == 1:
    axes = [axes]

for ax, cliente in zip(axes, clientes_sample):
    data = df_seq_pd[df_seq_pd['cliente_id'] == cliente]
    ax.plot(
        data['nro_pedido'],
        data['proporcion_digital_hist'],
        marker=None,
        color='gray'
    )
    ax.plot(
        data['nro_pedido'],
        data['es_digital'],
        marker='o',
        drawstyle='steps-mid',
        color='tab:blue',
        alpha=0.7,
        label='Es Digital'
    )
    ax.set_ylim(-0.1, 1.1)
    ax.set_title(f'Cliente {cliente}')
    ax.set_ylabel('Proporción Digital Histórica')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Número de Pedido')
plt.tight_layout()
plt.show()

## Creación del dataset

In [0]:
wind_cliente_all = Window.partitionBy("cliente_id").orderBy("fecha_pedido")
wind_cliente_6p = wind_cliente_all.rowsBetween(-5, Window.currentRow)
wind_cliente_12p = wind_cliente_all.rowsBetween(-11, Window.currentRow)

df_cliente = (
    df
    .withColumn("es_digital", (col("canal_pedido") == "DIGITAL").cast("int"))
    .withColumn("prop_digital_6p", f.avg("es_digital").over(wind_cliente_6p))
    .withColumn("prop_digital_12p", f.avg("es_digital").over(wind_cliente_12p))
    .withColumn("tendencia_digtal", col("prop_digital_6p") - f.lag("prop_digital_6p", 5).over(wind_cliente_all))
    .withColumn("tendencia_digtal", f.last("tendencia_digtal").over(wind_cliente_all))
    .withColumn("prop_digital_12p", f.last("prop_digital_12p").over(wind_cliente_all))
    .groupBy("cliente_id")
    .agg(
        f.last("agencia_id").alias("agencia_id"),
        f.last("ruta_id").alias("ruta_id"),
        f.last("pais").alias("pais"),
        f.last("region_comercial").alias("region_comercial"),
        f.last("tipo_cliente").alias("tipo_cliente"),
        f.last("madurez_digital").alias("madurez_digital"),
        f.last("estrellas").alias("estrellas"),
        f.last("frecuencia_visitas").alias("frecuencia_visitas"),
        f.sum("facturacion_usd").alias("sum_facturacion_usd"),
        f.avg("materiales_distintos").alias("avg_materiales_distintos"),
        f.sum("cajas_fisicas").alias("sum_cajas_fisicas"),
        f.count("*").alias("total_pedidos"),
        f.last("prop_digital_12p").alias("proporcion_digital"),
        f.last("tendencia_digtal").alias("tendencia_digital")
    )
)

display(df_cliente.limit(10))

In [0]:
df_pd = df_cliente.toPandas()

cat_cols = ['pais', 'region_comercial', 'tipo_cliente', 'frecuencia_visitas']
ordinal_cols = ['estrellas', 'madurez_digital']
num_cols = ['sum_facturacion_usd', 'avg_materiales_distintos', 'sum_cajas_fisicas', 'pedidos_digitales', 'total_pedidos']

for col_name in cat_cols:
    df_pd[col_name] = df_pd[col_name].astype("category")

df_pd['madurez_digital'] = pd.Categorical(
    df_pd['madurez_digital'],
    categories=["BAJA", "MEDIA", "ALTA"],
    ordered=True
)

df_pd['estrellas'] = pd.Categorical(
    df_pd['estrellas'],
    categories=[1, 2, 3],
    ordered=True
)

print('Dataset original:', f"{df.count():,}")
print('Dataset a nivel cliente:', f"{df_cliente.count():,}")
print('Dataset a nivel cliente (pandas):', f"{len(df_pd):,}")

display(df_pd)

## Análisis de variables

In [0]:
plt.figure(figsize=(8, 5))
sns.histplot(
    data=df_pd,
    x='proporcion_digital',
    bins=15
)
plt.title('Distribución de la Proporción Digital')
plt.xlabel('Proporción de Pedidos Digitales')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(8, 5))
sns.histplot(
    data=df_pd,
    x='tendencia_digital',
    bins=7
)
plt.title('Distribución de la Tendencia Digital')
plt.xlabel('Tendencia Digitales')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(8, 5))
sns.histplot(
    data=df_pd,
    x='total_pedidos',
    bins=15
)
plt.title('Distribución de los Pedidos Totales')
plt.xlabel('Total de Pedidos')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [0]:
import pandas as pd

# Definir los bins y etiquetas
bins = [0, 3, 5, 7, 9, 11, 13, float('inf')]
labels = ['1-3', '4-5', '6-7', '8-9', '10-11', '12-13', '14+']

df_pd['grupo_pedidos'] = pd.cut(df_pd['total_pedidos'], bins=bins, labels=labels, right=True)

plt.figure(figsize=(8, 5))
sns.countplot(
    data=df_pd,
    x='grupo_pedidos',
    order=labels
)
plt.title('Distribución de Clientes por Grupos de Total de Pedidos')
plt.xlabel('Grupo de Total de Pedidos')
plt.ylabel('Cantidad de Clientes')
plt.tight_layout()
plt.show()

In [0]:
# Definir los bins y etiquetas para tendencia digital
bins = [-float('inf'), -0.20, 0.20, float('inf')]
labels = ['Bajando', 'Sin Tendencia', 'Subiendo']

df_pd['grupo_tendencia'] = pd.cut(df_pd['tendencia_digital'], bins=bins, labels=labels, right=True)

plt.figure(figsize=(8, 5))
sns.countplot(
    data=df_pd,
    x='grupo_tendencia',
    order=labels
)
plt.title('Distribución de Clientes por Grupos de Tendencia Digital')
plt.xlabel('Grupo de Tendencia Digital')
plt.ylabel('Cantidad de Clientes')
plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(len(cat_cols + ordinal_cols), 1, figsize=(10, 6 * len(cat_cols + ordinal_cols)))

for ax, col_name in zip(axes, cat_cols + ordinal_cols):
    sns.boxplot(
        data=df_pd,
        x=col_name,
        y="proporcion_digital",
        hue=col_name,
        ax=ax
    )
    ax.set_title(f"{col_name.replace('_', ' ').title()} vs Proporción Digital")
    ax.set_xlabel(col_name.replace('_', ' ').title())
    ax.set_ylabel("Proporción de Pedidos Digitales")

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(len(num_cols), 1, figsize=(10, 6 * len(num_cols)))

for ax, col_name in zip(axes, num_cols):
    sns.kdeplot(
        data=df_pd,
        x=col_name,
        y="proporcion_digital",
        fill=True,
        cmap="Blues",
        ax=ax
    )
    ax.set_title(f"{col_name.replace('_', ' ').title()} vs Proporción Digital")
    ax.set_xlabel(col_name.replace('_', ' ').title())
    ax.set_ylabel("Proporción de Pedidos Digitales")

plt.tight_layout()
plt.show()

## Solución BI

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

sns.boxplot(
    data=df_pd,
    x="madurez_digital",
    y="proporcion_digital",
    hue="madurez_digital",
    palette=['#e74c3c', '#f39c12', '#3498db'],
    ax=ax
)

q3_baja = df_pd[df_pd["madurez_digital"] == "BAJA"]["proporcion_digital"].quantile(0.75)
q1_media = df_pd[df_pd["madurez_digital"] == "MEDIA"]["proporcion_digital"].quantile(0.25)
q3_media = df_pd[df_pd["madurez_digital"] == "MEDIA"]["proporcion_digital"].quantile(0.75)
q1_alta = df_pd[df_pd["madurez_digital"] == "ALTA"]["proporcion_digital"].quantile(0.25)

frontera_baja_media = (q3_baja + q1_media) / 2
frontera_media_alta = (q3_media + q1_alta) / 2

print("frontera_baja_media:", frontera_baja_media)
print("frontera_media_alta:", frontera_media_alta)

ax.axhline(frontera_baja_media, color="gray", linestyle="--")
ax.axhline(frontera_media_alta, color="gray", linestyle="--")

ax.set_title("Madurez Digital vs Proporción Digital")
ax.set_xlabel("Madurez Digital")
ax.set_ylabel("Proporción de Pedidos Digitales")

plt.tight_layout()
plt.show()

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

sns.boxplot(
    data=df_pd,
    x="madurez_digital",
    y="proporcion_digital",
    hue="grupo_tendencia",
    palette=['#e74c3c', '#f39c12', '#3498db'],
    ax=ax
)

ax.axhline(frontera_baja_media, color="gray", linestyle="--")
ax.axhline(frontera_media_alta, color="gray", linestyle="--")

ax.set_title("Madurez Digital vs Proporción Digital")
ax.set_xlabel("Madurez Digital")
ax.set_ylabel("Proporción de Pedidos Digitales")

plt.tight_layout()
plt.show()

## Clusterizacion

In [0]:
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Selección de variables
X = df_pd[["madurez_digital", "proporcion_digital", "tendencia_digital"]].copy()

# Eliminar filas con valores nulos
X = X.dropna()

# Codificar madurez_digital
X["madurez_digital"] = X["madurez_digital"].map({"BAJA": 0, "MEDIA": 1, "ALTA": 2})

# Estandarizar variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular Calinski-Harabasz y Davies-Bouldin para diferentes valores de k
calinski_scores = []
davies_scores = []
K = range(2, 12)

for k in K:
    print(f"Calculando para k={k}")
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    calinski_scores.append(calinski_harabasz_score(X_scaled, labels))
    davies_scores.append(davies_bouldin_score(X_scaled, labels))

fig, ax1 = plt.subplots(figsize=(12, 5))

color1 = 'tab:blue'
ax1.set_xlabel('Número de Clusters (k)')
ax1.set_ylabel('Calinski-Harabasz', color=color1)
ax1.plot(K, calinski_scores, marker='o', color=color1, label='Calinski-Harabasz')
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.set_ylabel('Davies-Bouldin', color=color2)
ax2.plot(K, davies_scores, marker='s', color=color2, label='Davies-Bouldin')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Calinski-Harabasz y Davies-Bouldin vs Número de Clusters')
fig.tight_layout()
plt.show()

In [0]:
# KMeans con k=5
X = df_pd[["madurez_digital", "proporcion_digital", "tendencia_digital"]].copy()
X = X.dropna()
X["madurez_digital"] = X["madurez_digital"].map({"BAJA": 0, "MEDIA": 1, "ALTA": 2})

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=5, random_state=24, n_init=10)
df_pd.loc[X.index, "cluster"] = kmeans.fit_predict(X_scaled).astype(int)

# Boxplot: cluster vs proporcion_digital
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_pd, y="cluster", x="proporcion_digital", palette="Set2", orient="h")
plt.title("Cluster vs Proporción Digital")
plt.xlabel("Cluster")
plt.ylabel("Proporción Digital")
plt.tight_layout()
plt.show()

# Boxplot: cluster vs tendencia_digital
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_pd, y="cluster", x="tendencia_digital", palette="Set2", orient="h")
plt.title("Cluster vs Tendencia Digital")
plt.xlabel("Cluster")
plt.ylabel("Tendencia Digital")
plt.tight_layout()
plt.show()

# 100% stacked bar: cluster vs madurez_digital
cluster_madurez = pd.crosstab(df_pd["cluster"], df_pd["madurez_digital"], normalize='index') * 100
cluster_madurez = cluster_madurez[["BAJA", "MEDIA", "ALTA"]]  # Ordenar columnas si es necesario

cluster_madurez.index = [f"Cluster {i}" for i in range(5)]

cluster_madurez.plot(
    kind='barh',
    stacked=True,
    color=['#e74c3c', '#f39c12', '#3498db'],
    figsize=(10, 6)
)
plt.title("Distribución de Madurez Digital por Cluster (100% Stacked)")
plt.xlabel("Porcentaje (%)")
plt.ylabel("Cluster (0 a 4)")
plt.legend(title="Madurez Digital", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()